Structured output



Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

Pydantic


Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [5]:
import os
from dotenv import load_dotenv

load_dotenv()
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:openai/gpt-oss-120b")


In [6]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movies rating out of 10")

In [7]:
model_with_structure=model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.6', 'langchain': '1.3.15'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000018DDA4E70E0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000018DDA4E7B60>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The t

In [8]:
res = model_with_structure.invoke("provide info about the movie 3 idiots")
res

Movie(title='3 Idiots', year=2009, director='Rajkumar Hirani', rating=8.4)

without parsed ouput with parsed output 
** include_raw=True**

In [9]:
model_with_structure_both=model.with_structured_output(Movie, include_raw=True)
model_with_structure_both

{
  raw: _ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.6', 'langchain': '1.3.15'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000018DDA4E70E0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000018DDA4E7B60>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description

In [10]:
res_both = model_with_structure_both.invoke("info about the dangal movie")
res_both

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks: "info about the dangal movie". Likely they want details: director, rating, title, year. We have a function Movie that can be called with director, rating, title, year. We need to supply info about the movie "Dangal". So we should call the function with appropriate data. Dangal is a 2016 Indian biographical sports drama film directed by Nitesh Tiwari, starring Aamir Khan. Rating? Could be IMDb rating ~8.4/10. Provide rating out of 10. We\'ll call function with title "Dangal", director "Nitesh Tiwari", rating 8.4, year 2016.', 'tool_calls': [{'id': 'fc_37e94611-d15a-4649-8c80-5d932ee1ca10', 'function': {'arguments': '{"director":"Nitesh Tiwari","rating":8.4,"title":"Dangal","year":2016}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 199, 'prompt_tokens': 157, 'total_tokens': 356, 'completion_time': 0.419947719, 'completion_tokens_details': {'reasoning_

NESTED STRUCTURE

In [11]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: list[str]
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movies which are highest grossing in 2000")

In [12]:
response

MovieDetails(title=['Mission: Impossible 2'], year=2000, cast=[Actor(name='Tom Cruise', role='Ethan Hunt'), Actor(name='Thandie Newton', role='Nyah Nordoff-Hall'), Actor(name='Dougray Scott', role='Sean Ambrose')], genres=['Action', 'Adventure', 'Thriller'], budget=140000000.0)

TYPEDICT

TypedDict provides a simpler alternative using Python’s built-in typing, ideal when you don’t need runtime validation.

In [13]:
from typing_extensions import TypedDict,Annotated

class moviedict(TypedDict):
    """A MOVIE WITH DETAILS"""
    title: Annotated[str, ...,"the title of the movie"]
    year: Annotated[int, ...,"the year of the movie"]
    director: Annotated[str, ...,"the director of the movie"]
    rating: Annotated[float, ...,"the rating of the movie"]

model_with_typedict= model.with_structured_output(moviedict)
res = model_with_typedict.invoke("give details of movie avengers")
res    

{'director': 'Joss Whedon', 'rating': 8, 'title': 'The Avengers', 'year': 2012}

In [14]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Avengers")
response

{'budget': 220000000,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'},
  {'name': 'Chris Evans', 'role': 'Steve Rogers / Captain America'},
  {'name': 'Mark Ruffalo', 'role': 'Bruce Banner / Hulk'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Scarlett Johansson', 'role': 'Natasha Romanoff / Black Widow'},
  {'name': 'Jeremy Renner', 'role': 'Clint Barton / Hawkeye'},
  {'name': 'Tom Hiddleston', 'role': 'Loki'},
  {'name': 'Clark Gregg', 'role': 'Phil Coulson'},
  {'name': 'Stellan Skarsgård', 'role': 'Erik Selvig'},
  {'name': 'Cobie Smulders', 'role': 'Maria Hill'},
  {'name': 'Samuel L. Jackson', 'role': 'Nick Fury'}],
 'genres': ['Action', 'Adventure', 'Sci-Fi', 'Superhero'],
 'title': 'The Avengers',
 'year': 2012}

dataclasses

A data class is a class typically containing mainly data, although there aren’t really any restrictions. You create it using the @dataclass decorator

A Python dataclass is an easy way to create a class whose main purpose is to store data.

In [17]:
#Without dataclass:
class User:
    def __init__(self, name, age):
        self.name = name
        self.age = age

user = User("Vishu", 21)
print(user.name)

Vishu


In [18]:
#With dataclass:

from dataclasses import dataclass

@dataclass
class User:
    name: str
    age: int

user = User("Vishu", 21)
print(user.age)

21


In [21]:
## Dataclass

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person


agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

In [ ]:
| Feature                         | **Dataclass**           | **Pydantic**                  | **TypedDict**               |
| ------------------------------- | ----------------------- | ----------------------------- | --------------------------- |
| **Data structure**              | ✅ Yes                   | ✅ Yes                         | ✅ Yes                       |
| **Type hints**                  | ✅ Yes                   | ✅ Yes                         | ✅ Yes                       |
| **Runtime validation**          | ⚠️ Limited              | **✅ Strong**                  | ❌ No                        |
| **Access data**                 | `obj.name`              | `obj.name`                    | `obj["name"]`               |
| **Returns**                     | Class object            | Pydantic object               | Dictionary                  |
| **Validation**                  | Basic Python type hints | **Automatic validation**      | Only type checking by tools |
| **LangChain structured output** | ✅ Yes                   | **✅ Yes**                     | ✅ Yes                       |
| **Best for**                    | Simple data objects     | **Validated/production data** | Dictionary-like data        |
| **Example**                     | `ContactInfo(...)`      | `ContactInfo(...)`            | `{"name": "John"}`          |
